In [ ]:
import random
import copy
import torch
import pickle
import os
import matplotlib.pyplot as plt
import numpy as np

from collections import defaultdict

from causal_gym import AntMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *
from causal_rl.algo.imitation.gail.core_net import *
from causal_rl.algo.imitation.gail.causal_gail import *

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '4'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
num_steps = 1000
hidden_dims = {'O'}

In [ ]:
# for eval: corrupted W, O hidden
eval_env = AntMazePCH(env_id='antmaze-large-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=False)

In [ ]:
# load models
MODEL_PATH = '/home/et2842/causal/causalrl/models/csqil_k100_antlarge.pt'
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

causal_sqil_actor = ContinuousActor(
    num_inputs=ckpt['z_dim'],
    num_outputs=ckpt['action_dim'],
    hidden_size=ckpt['hidden_size_actor'],
    std=0.0,
    action_low=float(ckpt['action_bounds_low'].min()),
    action_high=float(ckpt['action_bounds_high'].max()),
    num_blocks=ckpt['num_blocks_actor'],
    dropout=ckpt['dropout_actor'],
    layernorm=ckpt['layernorm_actor'],
).to(device)

causal_sqil_actor.load_state_dict(ckpt['state_dict'])
causal_sqil_actor.eval()

causal_sqil_Z_trim = ckpt['Z_sets']
dims = ckpt['dims']
lookback = ckpt['lookback']

causal_sqil_encode, _, _ = build_windowed_z_encoder(causal_sqil_Z_trim, dims=dims, lookback=lookback)
causal_sqil_policy = make_gail_policy(causal_sqil_actor, causal_sqil_encode, device=device, deterministic=True)
causal_sqil_policies = make_shared_policy_dict(causal_sqil_policy)

In [ ]:
num_eval_eps = 1000

causal_sqil_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=causal_sqil_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True
)

len(causal_sqil_returns)

In [ ]:
causal_sqil_episode_rewards = defaultdict(float)
for rec in causal_sqil_returns:
    ep = rec['episode']
    causal_sqil_episode_rewards[ep] += float(rec['reward'])

causal_sqil_rewards = [causal_sqil_episode_rewards[e] for e in range(num_eval_eps)]
sum(causal_sqil_rewards) / num_eval_eps

In [ ]:
mean_reward = np.mean(causal_sqil_rewards)
std_reward = np.std(causal_sqil_rewards)

print(f"E[Y]          = {mean_reward:.4f}")
print(f"Std[Y]        = {std_reward:.4f}")
print(f"E[Y] ± Std[Y] = {mean_reward:.4f} ± {std_reward:.4f}")

In [ ]:
# success rate: % of episodes solved in under 1000 steps
ep_lengths = defaultdict(int)
for rec in causal_sqil_returns:
    ep_lengths[rec['episode']] += 1

lengths = np.array([ep_lengths[e] for e in range(num_eval_eps)])
successes = lengths < num_steps
success_rate = successes.mean()
se = np.sqrt(success_rate * (1 - success_rate) / num_eval_eps)

print(f"Success rate   = {100 * success_rate:.2f}% ({successes.sum()}/{num_eval_eps} episodes)")
print(f"Std error      = {100 * se:.2f}%")

In [ ]:
# successful episode lengths
success_lengths = lengths[successes]

if len(success_lengths) > 0:
    print(f"Successful episode lengths (n={len(success_lengths)}):")
    print(f"  Mean   = {np.mean(success_lengths):.2f}")
    print(f"  Std    = {np.std(success_lengths):.2f}")
    print(f"  Median = {np.median(success_lengths):.0f}")
    print(f"  Min    = {np.min(success_lengths)}")
    print(f"  Max    = {np.max(success_lengths)}")
    print(f"  25th%  = {np.percentile(success_lengths, 25):.0f}")
    print(f"  75th%  = {np.percentile(success_lengths, 75):.0f}")
else:
    print("No episodes were solved.")